# VortexDB — Semantic Search Demo

This notebook shows VortexDB doing real semantic search using sentence-transformers embeddings.

**What happens:**
1. Embed a small corpus of sentences with `all-MiniLM-L6-v2` (384-dim)
2. Insert all embeddings into VortexDB
3. Search with a natural language query — results are ranked by cosine similarity
4. Add metadata (topic, year) and run filtered search

In [ ]:
# Install if needed
# !pip install sentence-transformers vectordb --no-build-isolation

In [ ]:
import sys, os, shutil
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
from sentence_transformers import SentenceTransformer
import vectordb

## 1. Load model and embed corpus

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

corpus = [
    # machine learning
    ("Transformers use self-attention to model long-range dependencies.",        "ml",  2017),
    ("BERT is a bidirectional transformer pre-trained on masked language modeling.", "ml", 2018),
    ("GPT generates text by predicting the next token autoregressively.",        "ml",  2018),
    ("Diffusion models iteratively denoise random noise to generate images.",    "ml",  2020),
    ("Reinforcement learning from human feedback improves language model alignment.", "ml", 2022),
    ("Mixture of experts scales model capacity without proportional compute.",   "ml",  2021),
    # databases
    ("HNSW builds a hierarchical graph for approximate nearest-neighbor search.", "db", 2016),
    ("LSM trees buffer writes in memory and merge sorted runs on disk.",         "db",  2006),
    ("Write-ahead logging ensures durability by persisting changes before applying them.", "db", 1992),
    ("Vector databases index high-dimensional embeddings for similarity search.", "db", 2021),
    ("B-trees provide O(log n) lookup by keeping data sorted in balanced pages.", "db", 1970),
    ("MVCC allows concurrent reads and writes without locking by keeping multiple versions.", "db", 1981),
]

sentences = [c[0] for c in corpus]
topics    = [c[1] for c in corpus]
years     = [c[2] for c in corpus]
ids       = [f"doc-{i:02d}" for i in range(len(corpus))]

print(f"Encoding {len(sentences)} sentences...")
embeddings = model.encode(sentences, convert_to_numpy=True).astype(np.float32)
print(f"Embedding shape: {embeddings.shape}")

## 2. Insert into VortexDB

In [ ]:
DATA_DIR = "/tmp/vortexdb_demo"
shutil.rmtree(DATA_DIR, ignore_errors=True)

db = vectordb.open(DATA_DIR)
db.create_collection("papers", dimension=embeddings.shape[1], metric="cosine")

metadata = [{"topic": t, "year": float(y)} for t, y in zip(topics, years)]
db.insert("papers", ids=ids, vectors=embeddings, metadata=metadata)

print(f"Inserted {len(ids)} documents into 'papers' collection.")

## 3. Semantic search

In [ ]:
def search_and_print(query_text, top_k=5, filters=None):
    query_vec = model.encode([query_text], convert_to_numpy=True).astype(np.float32)[0]
    results = db.search("papers", query=query_vec, top_k=top_k,
                        ef_search=20, filters=filters)
    label = f"  filters={filters}" if filters else ""
    print(f"Query: \"{query_text}\"{label}")
    print("-" * 70)
    id_to_sent = dict(zip(ids, sentences))
    id_to_meta = dict(zip(ids, metadata))
    for r in results:
        meta = id_to_meta[r['id']]
        print(f"  [{r['id']}] dist={r['distance']:.4f}  "
              f"topic={meta['topic']}  year={int(meta['year'])}")
        print(f"    {id_to_sent[r['id']]}")
    print()

search_and_print("How do language models learn from human feedback?")

In [ ]:
search_and_print("How are writes made durable in a database?")

In [ ]:
search_and_print("efficient approximate nearest neighbor search")

## 4. Filtered search

Same query, but restrict results to a specific topic or year range.

In [ ]:
# Only ML papers
search_and_print("scaling model capacity",
                 top_k=4, filters={"topic": "ml"})

In [ ]:
# Only papers from 2020 or later
search_and_print("scaling model capacity",
                 top_k=4, filters={"year": {"$gte": 2020.0}})

In [ ]:
# ML papers from 2020 or later (combined AND)
search_and_print("scaling model capacity",
                 top_k=4, filters={"topic": "ml", "year": {"$gte": 2020.0}})

## 5. Delete and verify

In [ ]:
# Remove the HNSW paper and verify it no longer appears
hnsw_id = "doc-06"
db.remove("papers", hnsw_id)

results = db.search("papers",
                    query=model.encode(["approximate nearest neighbor graph"],
                                       convert_to_numpy=True).astype(np.float32)[0],
                    top_k=5)
assert all(r["id"] != hnsw_id for r in results), "deleted doc still appearing!"
print(f"{hnsw_id} successfully removed. Top result is now:")
print(f"  [{results[0]['id']}] {sentences[ids.index(results[0]['id'])]}")

## 6. Checkpoint and reopen

In [ ]:
db.checkpoint("papers")
del db

db2 = vectordb.open(DATA_DIR)
results = db2.search("papers",
                     query=model.encode(["language model alignment"],
                                        convert_to_numpy=True).astype(np.float32)[0],
                     top_k=3)
print("After checkpoint + reopen:")
id_to_sent = dict(zip(ids, sentences))
for r in results:
    print(f"  [{r['id']}] dist={r['distance']:.4f}  {id_to_sent[r['id']]}")